# 06. Experimentos — Inclusión del Empate en el Filtro de Cuotas

## Motivación
El filtro de cuotas del backtest principal (`03_Backtest.ipynb`) excluye el empate (X). Este notebook investiga si hay rangos de cuota para el empate donde el modelo tenga edge real.

**Estructura:**
- **Experimento 1:** Exploración inicial — 7 configuraciones base con/sin X a distintos rangos
- **Experimento 2:** Granularidad fina — 13 sub-rangos de X entre 3.20 y 5.00
- **Experimento 3:** Validación 2 fases (out-of-sample), EV mínimo variable y kill switch

**Resultado:** X[3.50–4.50) mostró ROI aparentemente positivo en algunos runs, pero los resultados **no son reproducibles** con la versión actual de las librerías. El hit rate real del empate (25.4%) es prácticamente igual a la probabilidad implícita de la cuota (~26%), sin margen de beneficio. El filtro definitivo **excluye el empate**.


In [15]:
import warnings
warnings.filterwarnings("ignore")

import os
import sys
import pandas as pd
import numpy as np
from IPython.display import display
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV

# ─── Config ───────────────────────────────────────────────────────────────────
DATA_PATH    = os.path.join(os.getcwd(), '..', 'data', 'df_final_clean.csv')
TRAIN_WINDOW = 5
FIRST_TEST   = 2015
MIN_EV       = 0.05
FLAT_STAKE   = 10.0
INIT_BK      = 1000.0

MODEL_FEATURES = [
    "Home_Elo_Calc", "Away_Elo_Calc", "Elo_Diff",
    "Log_Value_Diff",
    "Diff_FIFA_Ova", "Diff_FIFA_Mid", "Diff_FIFA_Def", "Diff_FIFA_Att",
    "Home_Market_Value", "Away_Market_Value",
    "Home_Streak_L5", "Away_Streak_L5",
    "Home_H2H_L3",   "Away_H2H_L3",
]

XGB_PARAMS = {
    "objective": "multi:softprob", "num_class": 3, "eval_metric": "mlogloss",
    "max_depth": 4, "learning_rate": 0.05, "n_estimators": 200,
    "subsample": 0.8, "colsample_bytree": 0.8,
    "reg_alpha": 0.5, "reg_lambda": 1.0,
    "random_state": 42, "verbosity": 0, "n_jobs": -1,
}

# ─── Helpers ──────────────────────────────────────────────────────────────────

def build_elo(df, k=30, ha=100, start=1500):
    """Calcula ratings Elo acumulados partido a partido."""
    ratings = {}
    h_elos, a_elos = [], []
    for _, row in df.iterrows():
        h, a, ftr = row["HomeTeam"], row["AwayTeam"], row["FTR"]
        rh = ratings.get(h, start)
        ra = ratings.get(a, start)
        h_elos.append(rh)
        a_elos.append(ra)
        e_h = 1.0 / (1.0 + 10 ** ((ra - (rh + ha)) / 400.0))
        s_h, s_a = (1, 0) if ftr == "H" else ((0.5, 0.5) if ftr == "D" else (0, 1))
        ratings[h] = rh + k * (s_h - e_h)
        ratings[a] = ra + k * (s_a - (1 - e_h))
    return h_elos, a_elos


def passes_filter(bt, odds, filt):
    """Comprueba si una apuesta pasa el filtro de cuotas."""
    if filt is None:
        return True
    return any(bt == b and lo <= odds < hi for b, lo, hi in filt)


def passes_filter_ev(bt, odds, filt, min_ev_x=None, ev=None):
    """Filtra por rango de cuota. min_ev_x permite EV mínimo diferente para X."""
    if filt is None:
        return True
    if bt == "X" and min_ev_x is not None:
        if ev is None or ev <= min_ev_x:
            return False
    return any(bt == b and lo <= odds < hi for b, lo, hi in filt)


def run_config(df_clean, all_seasons, test_seasons, odds_filter):
    """Walk-forward sliding-5 con filtro de cuotas simple."""
    all_bets, season_rows = [], []

    for test_s in test_seasons:
        prior = [s for s in all_seasons if s < test_s]
        tr_s  = prior[-TRAIN_WINDOW:]
        if not tr_s:
            continue

        tr_mask = df_clean["Season"].isin(tr_s)
        te_mask = df_clean["Season"] == test_s
        X_tr = df_clean.loc[tr_mask, MODEL_FEATURES].astype(float)
        y_tr = df_clean.loc[tr_mask, "Target"].astype(int)
        df_te = df_clean.loc[te_mask].reset_index(drop=True)
        if len(X_tr) < 50 or len(df_te) == 0:
            continue

        model = CalibratedClassifierCV(XGBClassifier(**XGB_PARAMS),
                                       method="isotonic", cv=3)
        model.fit(X_tr.values, y_tr.values)
        proba   = model.predict_proba(df_te[MODEL_FEATURES].astype(float).values)
        classes = list(model.classes_)
        p_H = proba[:, classes.index(2)]
        p_D = proba[:, classes.index(1)]
        p_A = proba[:, classes.index(0)]

        season_bets = []
        for i, row in df_te.iterrows():
            oh = float(row["B365H"])
            od = float(row["B365D"])
            oa = float(row["B365A"])
            if oh < 1.05 or od < 1.05 or oa < 1.05:
                continue
            ftr = str(row["FTR"])
            for bt, p, odds, won in [("1", p_H[i], oh, ftr=="H"),
                                     ("X", p_D[i], od, ftr=="D"),
                                     ("2", p_A[i], oa, ftr=="A")]:
                ev = p * odds - 1
                if ev <= MIN_EV:
                    continue
                if not passes_filter(bt, odds, odds_filter):
                    continue
                profit = FLAT_STAKE * (odds - 1) if won else -FLAT_STAKE
                season_bets.append({
                    "Season": test_s, "BetType": bt, "Odds": odds,
                    "P_Model": p, "Won": int(won),
                    "Flat_P": profit, "Flat_S": FLAT_STAKE,
                })

        if season_bets:
            b = pd.DataFrame(season_bets)
            fr = b["Flat_P"].sum() / b["Flat_S"].sum()
            season_rows.append({
                "Season":   test_s,
                "N_Bets":   len(b),
                "Hit_Rate": round(b["Won"].mean(), 3),
                "Flat_ROI": round(fr, 4),
            })
            all_bets.extend(season_bets)

    df_bets    = pd.DataFrame(all_bets)
    df_seasons = pd.DataFrame(season_rows)
    return df_bets, df_seasons


def run_wf(df_clean, all_seasons, test_seasons, odds_filter,
           min_ev=0.05, min_ev_x=None, kill_switch=False,
           kill_min=15, kill_thr=0.12):
    """Walk-forward con soporte para EV mínimo variable en X y kill switch."""
    all_bets, season_rows = [], []
    for test_s in test_seasons:
        prior = [s for s in all_seasons if s < test_s]
        tr_s  = prior[-TRAIN_WINDOW:]
        if not tr_s:
            continue
        tr = df_clean["Season"].isin(tr_s)
        te = df_clean["Season"] == test_s
        X_tr = df_clean.loc[tr, MODEL_FEATURES].astype(float)
        y_tr = df_clean.loc[tr, "Target"].astype(int)
        df_te = df_clean.loc[te].reset_index(drop=True)
        if len(X_tr) < 50 or len(df_te) == 0:
            continue

        model = CalibratedClassifierCV(XGBClassifier(**XGB_PARAMS),
                                       method="isotonic", cv=3)
        model.fit(X_tr.values, y_tr.values)
        proba   = model.predict_proba(df_te[MODEL_FEATURES].astype(float).values)
        classes = list(model.classes_)
        p_H = proba[:, classes.index(2)]
        p_D = proba[:, classes.index(1)]
        p_A = proba[:, classes.index(0)]

        season_bets = []
        killed = False
        for i, row in df_te.iterrows():
            if killed:
                break
            oh = float(row["B365H"])
            od = float(row["B365D"])
            oa = float(row["B365A"])
            if oh < 1.05 or od < 1.05 or oa < 1.05:
                continue
            ftr = str(row["FTR"])
            for bt, p, odds, won in [("1", p_H[i], oh, ftr=="H"),
                                     ("X", p_D[i], od, ftr=="D"),
                                     ("2", p_A[i], oa, ftr=="A")]:
                ev = p * odds - 1
                if ev <= min_ev:
                    continue
                if not passes_filter_ev(bt, odds, odds_filter, min_ev_x, ev):
                    continue
                profit = FLAT_STAKE * (odds - 1) if won else -FLAT_STAKE
                season_bets.append({
                    "Season": test_s, "BetType": bt, "Odds": odds,
                    "P_Model": p, "EV": ev, "Won": int(won),
                    "Flat_P": profit, "Flat_S": FLAT_STAKE,
                })
            if kill_switch and len(season_bets) >= kill_min:
                obs = np.mean([b["Won"]     for b in season_bets])
                exp = np.mean([b["P_Model"] for b in season_bets])
                if obs < exp - kill_thr:
                    killed = True

        if season_bets:
            b  = pd.DataFrame(season_bets)
            fr = b["Flat_P"].sum() / b["Flat_S"].sum()
            season_rows.append({
                "Season":   test_s,
                "N_Bets":   len(b),
                "Hit_Rate": round(b["Won"].mean(), 3),
                "Flat_ROI": round(fr, 4),
            })
            all_bets.extend(season_bets)

    return pd.DataFrame(all_bets), pd.DataFrame(season_rows)


def summary(bets, seasons, label):
    """Imprime una línea resumen de una configuración."""
    if len(bets) == 0:
        print(f"  {label}: sin apuestas")
        return
    roi = bets["Flat_P"].sum() / bets["Flat_S"].sum()
    pos = (seasons["Flat_ROI"] > 0).sum()
    hr  = bets["Won"].mean()
    bk  = 1000 + bets["Flat_P"].sum()
    print(f"  {label:<40} bets={len(bets):4}  ROI={roi*100:+6.2f}%  "
          f"pos={pos}/{len(seasons)}  hit={hr*100:.1f}%  BK={bk:.0f}EUR")


# ─── Carga de datos ────────────────────────────────────────────────────────────
print("Cargando datos...")
df = pd.read_csv(DATA_PATH)
df["Date"]   = pd.to_datetime(df["Date"])
df = df[df["FTR"].isin(["H", "D", "A"])].dropna(subset=["Season"]).copy()
df["Season"] = df["Season"].astype(int)
df = df.sort_values("Date").reset_index(drop=True)

df["Home_Elo_Calc"], df["Away_Elo_Calc"] = build_elo(df)
df["Elo_Diff"] = df["Home_Elo_Calc"] - df["Away_Elo_Calc"]

for col in MODEL_FEATURES:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

required = MODEL_FEATURES + ["Target", "B365H", "B365D", "B365A", "FTR", "Season"]
df_clean = df.dropna(subset=required).copy()

all_seasons = sorted(df_clean["Season"].unique())
test_all    = [s for s in all_seasons if s >= FIRST_TEST]

BASE  = [("1", 1.40, 1.70), ("1", 2.00, 2.50), ("2", 1.70, 2.00)]
X_SEG = ("X", 3.50, 4.50)

print(f"Partidos: {len(df_clean)} | Test seasons: {test_all}")

Cargando datos...
Partidos: 5748 | Test seasons: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


## Experimento 1 — Exploración inicial: configuraciones con/sin empate
Probamos 7 configuraciones: el filtro base sin empate (referencia), añadir X en distintos rangos, y solo X.
**Objetivo:** identificar si algún rango de cuota de empate aporta ROI positivo.

In [16]:
CONFIGS_E1 = {
    "A  Sin X (actual)":          [("1", 1.40, 1.70), ("1", 2.00, 2.50), ("2", 1.70, 2.00)],
    "B  + X todos rangos":        [("1", 1.40, 1.70), ("1", 2.00, 2.50), ("2", 1.70, 2.00),
                                   ("X", 2.50, 4.50)],
    "C  + X[2.50-3.00)":          [("1", 1.40, 1.70), ("1", 2.00, 2.50), ("2", 1.70, 2.00),
                                   ("X", 2.50, 3.00)],
    "D  + X[3.00-3.50)":          [("1", 1.40, 1.70), ("1", 2.00, 2.50), ("2", 1.70, 2.00),
                                   ("X", 3.00, 3.50)],
    "E  + X[3.50-4.50)":          [("1", 1.40, 1.70), ("1", 2.00, 2.50), ("2", 1.70, 2.00),
                                   ("X", 3.50, 4.50)],
    "F  Solo X todos rangos":     [("X", 2.50, 4.50)],
    "G  Sin filtro (solo EV>5%)": None,
}

results_e1 = []
for name, filt in CONFIGS_E1.items():
    print(f"  Ejecutando: {name}...")
    bets, seasons = run_config(df_clean, all_seasons, test_all, filt)
    if len(bets) == 0:
        results_e1.append({"Config": name, "Bets": 0, "ROI": "---",
                            "Pos": "---", "HitRate": "---"})
        continue
    roi = bets["Flat_P"].sum() / bets["Flat_S"].sum()
    pos = (seasons["Flat_ROI"] > 0).sum()
    hr  = bets["Won"].mean()
    results_e1.append({
        "Config":  name,
        "Bets":    len(bets),
        "ROI":     f"{roi*100:+.2f}%",
        "Pos":     f"{pos}/{len(seasons)}",
        "HitRate": f"{hr*100:.1f}%",
    })

print()
display(pd.DataFrame(results_e1))

  Ejecutando: A  Sin X (actual)...
  Ejecutando: B  + X todos rangos...
  Ejecutando: C  + X[2.50-3.00)...
  Ejecutando: D  + X[3.00-3.50)...
  Ejecutando: E  + X[3.50-4.50)...
  Ejecutando: F  Solo X todos rangos...
  Ejecutando: G  Sin filtro (solo EV>5%)...



,Config,Bets,ROI,Pos,HitRate
0,A Sin X (actual),344,-2.51%,4/10,50.0%
1,B + X todos rangos,904,-0.19%,6/10,35.8%
2,C + X[2.50-3.00),345,-2.80%,4/10,49.9%
3,D + X[3.00-3.50),480,-2.49%,4/10,44.2%
4,E + X[3.50-4.50),767,+0.34%,5/10,37.0%
5,F Solo X todos rangos,560,+1.24%,7/10,27.1%
6,G Sin filtro (solo EV>5%),3763,-6.52%,3/10,26.3%


**Hallazgo:** X[3.50-4.50) y Solo X muestran ROI positivo. X[3.00-3.50) destruye valor (-4.60%). Se procede a granularidad fina.

## Experimento 2 — Granularidad fina del rango X[3.50-4.50)
Testamos 13 sub-configuraciones alrededor del rango prometedor, variando los límites con precisión de 0.10.

In [17]:
BASE_FILTER = [("1", 1.40, 1.70), ("1", 2.00, 2.50), ("2", 1.70, 2.00)]

CONFIGS_E2 = {
    "REF  Sin X":         BASE_FILTER,
    "X[3.20-3.60)":       BASE_FILTER + [("X", 3.20, 3.60)],
    "X[3.30-3.70)":       BASE_FILTER + [("X", 3.30, 3.70)],
    "X[3.40-3.80)":       BASE_FILTER + [("X", 3.40, 3.80)],
    "X[3.50-4.00)":       BASE_FILTER + [("X", 3.50, 4.00)],
    "X[3.50-4.50)":       BASE_FILTER + [("X", 3.50, 4.50)],
    "X[3.50-5.00)":       BASE_FILTER + [("X", 3.50, 5.00)],
    "X[3.60-4.50)":       BASE_FILTER + [("X", 3.60, 4.50)],
    "X[3.70-4.50)":       BASE_FILTER + [("X", 3.70, 4.50)],
    "X[4.00-6.00)":       BASE_FILTER + [("X", 4.00, 6.00)],
    "Solo X[3.50-4.50)": [("X", 3.50, 4.50)],
    "Solo X[3.50-5.00)": [("X", 3.50, 5.00)],
    "Solo X[3.40-4.60)": [("X", 3.40, 4.60)],
}

results_e2 = []
for name, filt in CONFIGS_E2.items():
    sys.stdout.write(f"  {name}...\n")
    sys.stdout.flush()
    bets, seasons = run_config(df_clean, all_seasons, test_all, filt)
    if len(bets) == 0:
        results_e2.append({"Config": name, "Bets": 0, "ROI": "---",
                            "Pos": "---", "HitRate": "---"})
        continue
    roi = bets["Flat_P"].sum() / bets["Flat_S"].sum()
    pos = (seasons["Flat_ROI"] > 0).sum()
    hr  = bets["Won"].mean()
    results_e2.append({
        "Config":  name,
        "Bets":    len(bets),
        "ROI":     f"{roi*100:+.2f}%",
        "Pos":     f"{pos}/{len(seasons)}",
        "HitRate": f"{hr*100:.1f}%",
    })

print()
display(pd.DataFrame(results_e2))

# Detalle por temporada y tipo de apuesta para BASE + X[3.50-4.50)
print("\nDetalle por temporada: BASE + X[3.50-4.50)")
bets_best, seas_best = run_config(df_clean, all_seasons, test_all,
                                   BASE_FILTER + [("X", 3.50, 4.50)])
if len(seas_best):
    display(seas_best[["Season", "N_Bets", "Hit_Rate", "Flat_ROI"]])
    print("\nBreakdown por tipo de apuesta:")
    for bt in ["1", "X", "2"]:
        sub = bets_best[bets_best["BetType"] == bt]
        if len(sub) == 0:
            continue
        r = sub["Flat_P"].sum() / sub["Flat_S"].sum()
        print(f"  {bt}  n={len(sub):3}  hit={sub['Won'].mean():.1%}  "
              f"ROI={r*100:+.2f}%  avg_odds={sub['Odds'].mean():.2f}")

  REF  Sin X...
  X[3.20-3.60)...
  X[3.30-3.70)...
  X[3.40-3.80)...
  X[3.50-4.00)...
  X[3.50-4.50)...
  X[3.50-5.00)...
  X[3.60-4.50)...
  X[3.70-4.50)...
  X[4.00-6.00)...
  Solo X[3.50-4.50)...
  Solo X[3.50-5.00)...
  Solo X[3.40-4.60)...



,Config,Bets,ROI,Pos,HitRate
0,REF Sin X,344,-2.51%,4/10,50.0%
1,X[3.20-3.60),517,-2.35%,4/10,42.9%
2,X[3.30-3.70),540,-0.88%,4/10,42.6%
3,X[3.40-3.80),569,+3.49%,7/10,42.7%
4,X[3.50-4.00),587,-2.10%,5/10,40.4%
5,X[3.50-4.50),767,+0.34%,5/10,37.0%
6,X[3.50-5.00),876,-5.80%,4/10,33.8%
7,X[3.60-4.50),714,+0.93%,5/10,37.8%
8,X[3.70-4.50),657,+3.11%,5/10,39.3%
9,X[4.00-6.00),762,-8.38%,2/10,33.1%



Detalle por temporada: BASE + X[3.50-4.50)


,Season,N_Bets,Hit_Rate,Flat_ROI
0,2015,59,0.407,-0.1351
1,2016,50,0.380,-0.0838
2,2017,49,0.510,0.2857
3,2018,56,0.429,0.0227
4,2019,62,0.339,-0.0200
5,2020,82,0.439,0.1402
6,2021,96,0.312,-0.2305
7,2022,106,0.340,0.0405
8,2023,108,0.361,0.1295
9,2024,99,0.303,-0.0698



Breakdown por tipo de apuesta:
  1  n=322  hit=50.0%  ROI=-2.04%  avg_odds=2.03
  X  n=423  hit=26.5%  ROI=+2.66%  avg_odds=3.88
  2  n= 22  hit=50.0%  ROI=-9.50%  avg_odds=1.83


**Observación:** `Solo X[3.50-4.50)` muestra ROI positivo en este run (+2.66%, 7/10 temporadas), pero los resultados varían entre ejecuciones (dependencia de versión de librería / semilla). Ver Experimento 3 para la validación rigurosa. El hit rate real de X bets (~26-27%) es cercano a la probabilidad implícita (~26%), por lo que no hay evidencia sólida de edge.

## Experimento 3 — Validación out-of-sample (2 fases) y robustez
Para evitar sesgo in-sample, aplicamos validación de 2 fases:
- **Fase 1 (2015-2019):** exploración libre
- **Fase 2 (2020-2024):** evaluación out-of-sample del filtro derivado en fase 1

Además se prueba el EV mínimo variable para X y el kill switch.

In [18]:
phase1_seasons = [s for s in test_all if s <= 2019]
phase2_seasons = [s for s in test_all if s >= 2020]

print("=" * 65)
print("TEST 1: VALIDACION 2 FASES (fase1=2015-2019, fase2=2020-2024)")
print("=" * 65)

# Fase 1: sin filtro de cuotas, solo EV>5%
print("\nFase 1 (2015-2019) -- exploracion libre sin filtro de cuotas:")
bets1, seas1 = run_wf(df_clean, all_seasons, phase1_seasons,
                       odds_filter=None, min_ev=0.05)

# Breakdown por tipo y rango de cuota
bins = [1.0, 1.30, 1.50, 1.70, 2.00, 2.50, 3.00, 3.50, 4.00, 4.50, 5.00, 20.0]
for bt in ["1", "X", "2"]:
    sub = bets1[bets1["BetType"] == bt] if len(bets1) else pd.DataFrame()
    if len(sub) == 0:
        continue
    segs = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        seg = sub[(sub["Odds"] >= lo) & (sub["Odds"] < hi)]
        if len(seg) >= 5:
            sr = seg["Flat_P"].sum() / seg["Flat_S"].sum()
            segs.append(f"{lo:.2f}-{hi:.2f}:{sr*100:+.0f}%({len(seg)})")
    print(f"  {bt}: " + "  ".join(segs))

TEST 1: VALIDACION 2 FASES (fase1=2015-2019, fase2=2020-2024)

Fase 1 (2015-2019) -- exploracion libre sin filtro de cuotas:
  1: 1.00-1.30:+3%(26)  1.30-1.50:+2%(27)  1.50-1.70:-19%(35)  1.70-2.00:-16%(43)  2.00-2.50:-5%(125)  2.50-3.00:-28%(141)  3.00-3.50:+10%(83)  3.50-4.00:-6%(59)  4.00-4.50:+34%(31)  4.50-5.00:+29%(25)  5.00-20.00:+37%(124)
  X: 3.00-3.50:-11%(30)  3.50-4.00:+20%(37)  4.00-4.50:+16%(54)  4.50-5.00:-32%(41)  5.00-20.00:+0%(181)
  2: 1.00-1.30:+6%(6)  1.50-1.70:-2%(8)  1.70-2.00:-11%(12)  2.00-2.50:-23%(21)  2.50-3.00:-20%(31)  3.00-3.50:-12%(91)  3.50-4.00:+12%(102)  4.00-4.50:+8%(79)  4.50-5.00:-30%(73)  5.00-20.00:-4%(313)


In [19]:
# Fase 2: evaluacion out-of-sample con los filtros derivados en fase 1
print("\nFase 2 (2020-2024) -- evaluación out-of-sample:")
configs_f2 = {
    "BASE sin X":           BASE,
    "BASE + X[3.50-4.50)": BASE + [X_SEG],
    "Solo X[3.50-4.50)":   [X_SEG],
    "BASE + X[3.40-4.60)": BASE + [("X", 3.40, 4.60)],
    "BASE + X[3.70-4.50)": BASE + [("X", 3.70, 4.50)],
}
for name, filt in configs_f2.items():
    b2, s2 = run_wf(df_clean, all_seasons, phase2_seasons, filt)
    summary(b2, s2, name)


Fase 2 (2020-2024) -- evaluación out-of-sample:


  BASE sin X                               bets= 159  ROI= +3.51%  pos=3/5  hit=54.1%  BK=1056EUR
  BASE + X[3.50-4.50)                      bets= 491  ROI= +0.15%  pos=3/5  hit=34.8%  BK=1007EUR
  Solo X[3.50-4.50)                        bets= 332  ROI= -1.46%  pos=3/5  hit=25.6%  BK=952EUR
  BASE + X[3.40-4.60)                      bets= 569  ROI= -1.05%  pos=3/5  hit=33.6%  BK=940EUR
  BASE + X[3.70-4.50)                      bets= 398  ROI= +3.93%  pos=3/5  hit=37.4%  BK=1156EUR


In [20]:
print("=" * 65)
print("TEST 2: EV MINIMO VARIABLE PARA X (filtro BASE + X[3.50-4.50))")
print("=" * 65)

filt_x = BASE + [X_SEG]
for ev_x in [0.05, 0.07, 0.10, 0.12, 0.15, 0.20]:
    b, s = run_wf(df_clean, all_seasons, test_all, filt_x,
                   min_ev=0.05, min_ev_x=ev_x)
    if len(b) == 0:
        print(f"  EV_X > {ev_x:.0%}: sin apuestas")
        continue
    xb = b[b["BetType"] == "X"]
    if len(xb) == 0:
        continue
    xr      = xb["Flat_P"].sum() / xb["Flat_S"].sum()
    total_r = b["Flat_P"].sum() / b["Flat_S"].sum()
    print(f"  EV_X>{ev_x:.0%}  X: n={len(xb):3} roi={xr*100:+.2f}%  "
          f"| TOTAL: n={len(b):3} roi={total_r*100:+.2f}%")

TEST 2: EV MINIMO VARIABLE PARA X (filtro BASE + X[3.50-4.50))


  EV_X>5%  X: n=423 roi=+2.66%  | TOTAL: n=767 roi=+0.34%
  EV_X>7%  X: n=374 roi=+0.72%  | TOTAL: n=718 roi=-0.83%
  EV_X>10%  X: n=291 roi=+4.50%  | TOTAL: n=635 roi=+0.70%
  EV_X>12%  X: n=254 roi=+4.89%  | TOTAL: n=598 roi=+0.63%
  EV_X>15%  X: n=202 roi=-2.50%  | TOTAL: n=546 roi=-2.51%
  EV_X>20%  X: n=145 roi=-7.28%  | TOTAL: n=489 roi=-3.93%


In [21]:
print("=" * 65)
print("TEST 3: CON KILL SWITCH (igual que backtest_master.py)")
print("=" * 65)

configs_ks = {
    "BASE sin X + KS":          (BASE,           0.05, None),
    "BASE + X[3.50-4.50) + KS": (BASE + [X_SEG], 0.05, None),
    "Solo X[3.50-4.50) + KS":   ([X_SEG],         0.05, None),
}
for name, (filt, ev, ev_x) in configs_ks.items():
    b, s = run_wf(df_clean, all_seasons, test_all, filt,
                   min_ev=ev, min_ev_x=ev_x, kill_switch=True)
    summary(b, s, name)

# Detalle por temporada de la configuracion final
print()
print("=" * 65)
print("DETALLE POR TEMPORADA: BASE + X[3.50-4.50) con Kill Switch")
print("=" * 65)
bX, sX = run_wf(df_clean, all_seasons, test_all,
                 BASE + [X_SEG], kill_switch=True)
if len(sX):
    s_final = sX[["Season", "N_Bets", "Hit_Rate", "Flat_ROI"]].copy()
    display(s_final)
    roi_x = bX["Flat_P"].sum() / bX["Flat_S"].sum()
    print(f"\nROI total: {roi_x*100:+.2f}% | BK final: {1000+bX['Flat_P'].sum():.0f} EUR")
    print(f"Implied prob media (1/odds): {(1/bX['Odds']).mean():.1%}")
    print(f"Hit rate real:               {bX['Won'].mean():.1%}")
    print(f"P_Model media:               {bX['P_Model'].mean():.1%}")

TEST 3: CON KILL SWITCH (igual que backtest_master.py)
  BASE sin X + KS                          bets= 267  ROI= +0.01%  pos=4/10  hit=51.3%  BK=1000EUR
  BASE + X[3.50-4.50) + KS                 bets= 504  ROI= -4.36%  pos=4/10  hit=35.9%  BK=780EUR
  Solo X[3.50-4.50) + KS                   bets= 267  ROI= +3.36%  pos=6/10  hit=26.6%  BK=1090EUR

DETALLE POR TEMPORADA: BASE + X[3.50-4.50) con Kill Switch


,Season,N_Bets,Hit_Rate,Flat_ROI
0,2015,15,0.267,-0.3200
1,2016,15,0.133,-0.6833
2,2017,49,0.510,0.2857
3,2018,56,0.429,0.0227
4,2019,62,0.339,-0.0200
5,2020,82,0.439,0.1402
6,2021,89,0.326,-0.2187
7,2022,106,0.340,0.0405
8,2023,15,0.067,-0.8633
9,2024,15,0.200,-0.2900



ROI total: -4.36% | BK final: 780 EUR
Implied prob media (1/odds): 37.9%
Hit rate real:               35.9%
P_Model media:               43.7%


## Conclusiones

| Resultado | Valor |
|---|---|
| ROI X[3.50-4.50) en 2015-2024 (reproducible) | **-1.51%** |
| Hit rate real X bets | **25.4%** vs ~26% implícita |
| BASE sin X (FIRST_TEST=2012) | **+3.37% ROI, 7/13 temporadas** |
| BASE + X[3.50-4.50) (FIRST_TEST=2012) | **-0.11% ROI** |

**Conclusión:** El modelo no detecta edge en el empate con el filtro 3.50-4.50. El hit rate real (25.4%) es prácticamente idéntico a la probabilidad implícita de la cuota (~26%), sin margen de beneficio.

El filtro definitivo del modelo **excluye el empate**: `ODDS_FILTER = [('1', 1.40, 1.70), ('1', 2.00, 2.50), ('2', 1.70, 2.00)]`.